# V3 Pipeline Restructure — Symmetric Exit / Return Detection

This notebook reads `flight_tracks.csv` and `detections.csv` produced by
`components.ipynb`, heals camera spikes without destroying tracks, and applies
**symmetric v3 logic** to both hive exits and returns.

**Why this exists.** The legacy v1 detector calls a track a "return" whenever
the bee passes within 0.1 m of the hive at any point in the last 10 frames.
That over-counts fly-bys (226 v1 returns vs. 152 v1 exits in our data). The
v3 detector adds three checks on top of v1:

1. **Terminal/initial speed** must drop below 0.5 m/s (biological landing /
   take-off speed).
2. **Velocity extrapolation** (forward for returns, backward for exits) must
   intersect a 0.15 m hive sphere.
3. **End/start position** of the track must actually be near the hive
   (≤ 0.2 m for returns, ≤ 0.3 m for exits).
4. **Approach-vector alignment**: the cosine between the velocity vector and
   the direction to the hive must be > 0.3 — kills perpendicular fly-bys.

Both `*_v1` and `*_v3` boolean columns are kept side-by-side so you can diff
them (`v1=True & v3=False` → candidates for the 3D fly-by visualiser).

**Pipeline:**
1. Load — read flight_tracks.csv + detections.csv (same path as `components.ipynb`).
2. Heal — null + linearly interpolate frames where velocity > 7 m/s, then smooth.
3. Detect exits (v1 + v3).
4. Detect returns (v1 + v3).
5. Pair v3 exits → next v3 return chronologically (≤ 60 min cap).
6. Log unmatched exits / returns and write everything to `data/.../v3_pipeline/`.


In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd


# CONFIG — every tunable lives here, the rest of the notebook just reads from it
# ---------------------------------------------------------------------------

# Which day / camera selection to read (must match what components.ipynb wrote).
# Folders on disk are always `YYYY-MM-DD_system_<id>`; there is no "both"
# folder, so SPOT_SELECTION must be "900" or "939".
TARGET_DATE     = datetime(2026, 5, 12)
SPOT_SELECTION  = "900"   # "900" or "939"

# Frame rate — components.ipynb uses 60.0
FPS = 60.0

# Sphere radii (metres)
TOL              = 0.10    # legacy v1: "did the bee come within this of the hive"
HIVE_SPHERE_V3   = 0.15    # v3: extrapolated line must pass within this of hive

HEAD_FRAMES = 10
TAIL_FRAMES = 10

# v3 speed thresholds (m/s)
SPEED_THRESHOLD_TAKEOFF = 0.5
SPEED_THRESHOLD_LANDING = 0.5

# v3 location thresholds (m)
END_NEAR_HIVE_MAX    = 0.20   # return: last point must be ≤ this from hive
START_NEAR_HIVE_MAX  = 0.30   # exit:   first point must be ≤ this from hive
                              # (softer because takeoff accelerates fast)

# Cosine alignment between velocity vector and (hive - position)
APPROACH_COS_MIN_RETURN = 0.30   # 1.0 = points directly at hive, 0.0 = perpendicular
APPROACH_COS_MIN_EXIT   = 0.30   # for exits we check the OUTWARD direction

# ---- v3 leniency knobs (added after first run was too strict) -------------
# Bees decelerate ACROSS the landing window, so the mean speed over 10 frames
# is misleadingly high. min-speed catches the actual landing/takeoff moment
# without letting in fly-bys (which maintain constant speed -> high min too).
USE_MIN_SPEED_FOR_THRESHOLD = True

# If the end (return) / start (exit) position is already inside the v3 hive
# sphere, the geometric extrapolation question is moot — auto-pass it.
AUTO_PASS_EXTRAPOLATION_AT_HIVE = True

# Below this mean-velocity magnitude (m/s) the direction vector is dominated
# by noise — skip the cosine alignment check and fall back to position checks.
HOVER_SPEED_MIN = 0.10

# Compute approach/outward DIRECTION from frames slightly outside the
# head/tail window (where the bee is moving more cleanly). For exits this
# means frames HEAD..HEAD+OFFSET; for returns frames -TAIL-OFFSET..-TAIL.
# Set to 0 to use the head/tail frames themselves.
APPROACH_DIR_FRAME_OFFSET = 10

# Spike healing
MAX_BIO_VELOCITY = 7.0    # m/s — anything above this is a camera glitch
SMOOTH_WINDOW    = 5      # centered rolling mean over 5 frames
MIN_TRACK_FRAMES = 10     # drop tracks with fewer valid frames after cleaning

# Pairing
MAX_TRIP_DURATION_S = 60 * 60   # 60 min cap on exit→return matching

# Hive positions per camera system (copied from components.ipynb)
HIVE_BY_SYSTEM = {
    900: np.array([-0.04,  -0.665, -1.195], dtype=float),
    939: np.array([-0.086, -0.828, -1.045], dtype=float),
}

# Resolve the data directory using the same convention as components.ipynb.
# Folders on disk are always `YYYY-MM-DD_system_<id>`.
_sel_label = f"system_{SPOT_SELECTION}"
DATA_DIR = (Path("..") / ".." / "data" / "flight_data"
            / f"{TARGET_DATE.strftime('%Y-%m-%d')}_{_sel_label}")

TRACKS_CSV     = DATA_DIR / "flight_tracks.csv"
DETECTIONS_CSV = DATA_DIR / "detections.csv"

print(f"Data dir:        {DATA_DIR}")
print(f"  flight_tracks: {TRACKS_CSV.exists()}")
print(f"  detections:    {DETECTIONS_CSV.exists()}")


In [ ]:

# Load
# ---------------------------------------------------------------------------

all_tracks = pd.read_csv(TRACKS_CSV)
detections = pd.read_csv(DETECTIONS_CSV)

# components.ipynb writes the uid column as "detection_uid" — normalise.
if "detection_uid" in all_tracks.columns and "uid" not in all_tracks.columns:
    all_tracks = all_tracks.rename(columns={"detection_uid": "uid"})

# uid -> system_id, for hive lookup
uid_to_system = (detections.set_index("uid")["system_id"]
                 .astype("Int64").to_dict())

# Track start / end times for chronological pairing.
# Try several conventions; fall back to frame / FPS (relative time).
def _resolve_times():
    if "time" in all_tracks.columns:
        starts = all_tracks.groupby("uid")["time"].min().to_dict()
        ends   = all_tracks.groupby("uid")["time"].max().to_dict()
        return starts, ends, "time column"
    for col in ("start_time", "timestamp", "datetime"):
        if col in detections.columns:
            t = pd.to_datetime(detections[col], errors="coerce").astype("int64") / 1e9
            starts = dict(zip(detections["uid"], t))
            return starts, dict(starts), f"detections.{col}"
    if "frame" in all_tracks.columns:
        f0 = all_tracks["frame"].min()
        starts = ((all_tracks.groupby("uid")["frame"].min() - f0) / FPS).to_dict()
        ends   = ((all_tracks.groupby("uid")["frame"].max() - f0) / FPS).to_dict()
        return starts, ends, f"frame/{FPS:.0f} (relative)"
    raise RuntimeError("No usable time/frame column found")

uid_to_start, uid_to_end, _time_source = _resolve_times()

print(f"Loaded {len(all_tracks):,} rows / "
      f"{all_tracks['uid'].nunique():,} tracks")
print(f"Systems: {sorted(detections['system_id'].dropna().astype(int).unique().tolist())}")
print(f"Track times derived from: {_time_source}")
all_tracks.head()


In [ ]:

# Heal velocity spikes + smooth jitter (per-track, non-destructive)
# ---------------------------------------------------------------------------

POS_COLS = ["posX_insect", "posY_insect", "posZ_insect"]


def heal_and_smooth_track(trk):
    """
    Clean a single uid's frames in-place (returns a new dataframe):
      1. Mask invalid samples (pos_valid_insect == 0).
      2. Detect frame-to-frame speeds > MAX_BIO_VELOCITY — nullify those points.
      3. Linearly interpolate the gaps.
      4. Centered rolling mean of width SMOOTH_WINDOW to take out residual jitter.
      5. Drop the track entirely if < MIN_TRACK_FRAMES survive.
    Adds two diagnostic columns: `is_interpolated`, `spike_flagged`.
    Returns None if the track is too short after cleaning.
    """
    trk = trk.copy().reset_index(drop=True)

    if "pos_valid_insect" in trk.columns:
        valid_mask = trk["pos_valid_insect"] == 1
    else:
        valid_mask = pd.Series(True, index=trk.index)

    coords = trk[POS_COLS].astype(float).copy()
    coords.loc[~valid_mask] = np.nan

    # Frame-to-frame speed (m/s)
    dt = 1.0 / FPS
    speed = np.linalg.norm(coords.diff().values, axis=1) / dt
    spike_mask = pd.Series(speed > MAX_BIO_VELOCITY, index=coords.index).fillna(False)

    # Nullify spike frames so they get interpolated over
    coords.loc[spike_mask.values] = np.nan
    interpolated_mask = spike_mask.values | (~valid_mask.values)

    # Linear interpolation in both directions (handles head/tail NaNs too)
    coords = coords.interpolate(method="linear", limit_direction="both")

    if coords.dropna().shape[0] < MIN_TRACK_FRAMES:
        return None

    # Centered rolling mean for jitter smoothing
    smoothed = (coords
                .rolling(window=SMOOTH_WINDOW, center=True, min_periods=1)
                .mean())

    out = trk.copy()
    out[POS_COLS] = smoothed.values
    out["is_interpolated"] = interpolated_mask
    out["spike_flagged"]   = spike_mask.values
    return out


cleaned_chunks = []
spike_counts   = []
dropped_uids   = []

for uid, trk in all_tracks.groupby("uid", sort=False):
    cleaned = heal_and_smooth_track(trk)
    if cleaned is None:
        dropped_uids.append(uid)
        continue
    cleaned_chunks.append(cleaned)
    spike_counts.append(int(cleaned["spike_flagged"].sum()))

if cleaned_chunks:
    cleaned_tracks = pd.concat(cleaned_chunks, ignore_index=True)
else:
    cleaned_tracks = pd.DataFrame(columns=list(all_tracks.columns) + ["is_interpolated", "spike_flagged"])

n_tracks_in = all_tracks["uid"].nunique()
print(f"Tracks in:   {n_tracks_in:,}")
print(f"Tracks out:  {len(cleaned_chunks):,}  (dropped {len(dropped_uids):,} short)")
print(f"Spike frames healed: {sum(spike_counts):,} across "
      f"{sum(1 for c in spike_counts if c > 0):,} tracks")


In [ ]:

# v3 exit detection
# ---------------------------------------------------------------------------


def line_passes_within_radius(point, direction, target, radius, require_forward=True):
    """
    Does the half-line starting at `point` with direction `direction` pass within
    `radius` of `target`?

    Returns (hits_sphere, t_at_closest_approach, closest_distance).
    `t_at_closest_approach` is the parameter along the direction vector
    (positive = forward). If require_forward and t < 0, returns hits_sphere=False.
    """
    d_norm = np.linalg.norm(direction)
    if d_norm < 1e-9:
        return False, np.nan, float("inf")
    direction = direction / d_norm
    to_target = target - point
    t = float(np.dot(to_target, direction))
    closest_point = point + t * direction
    closest_dist  = float(np.linalg.norm(closest_point - target))
    if require_forward and t < 0:
        return False, t, closest_dist
    return (closest_dist <= radius), t, closest_dist


def _valid_coords(trk):
    """Return (N, 3) array dropping rows where any coordinate is NaN."""
    arr = trk[POS_COLS].to_numpy(dtype=float)
    mask = ~np.isnan(arr).any(axis=1)
    return arr[mask]


def classify_exit_v3(trk, hive):
    coords = _valid_coords(trk)
    if len(coords) < HEAD_FRAMES:
        return None

    head = coords[:HEAD_FRAMES]
    dt = 1.0 / FPS

    # --- speed check (the bee was at rest at some point in the head) ---
    head_velocities = np.diff(head, axis=0) / dt
    head_speeds     = np.linalg.norm(head_velocities, axis=1)
    initial_speed_mean = float(np.mean(head_speeds))
    initial_speed_min  = float(np.min(head_speeds))
    speed_for_threshold = (initial_speed_min if USE_MIN_SPEED_FOR_THRESHOLD
                           else initial_speed_mean)
    is_taking_off = speed_for_threshold < SPEED_THRESHOLD_TAKEOFF

    # --- direction estimate: use slightly later frames if track is long enough,
    #     since the bee has accelerated past the noise floor by then ----------
    dir_end = min(HEAD_FRAMES + APPROACH_DIR_FRAME_OFFSET, len(coords))
    if APPROACH_DIR_FRAME_OFFSET > 0 and (dir_end - HEAD_FRAMES) >= 2:
        dir_window = coords[HEAD_FRAMES:dir_end]
        dir_velocities = np.diff(dir_window, axis=0) / dt
    else:
        dir_velocities = head_velocities
    mean_velocity = dir_velocities.mean(axis=0)
    speed_norm    = float(np.linalg.norm(mean_velocity))
    velocity_unit = mean_velocity / speed_norm if speed_norm >= 1e-9 else np.zeros(3)

    # --- backward extrapolation: line from head[0] in -velocity should hit hive
    hits_sphere, t_close, dist_close = line_passes_within_radius(
        point=head[0], direction=-velocity_unit, target=hive,
        radius=HIVE_SPHERE_V3, require_forward=True,
    )

    start_dist_to_hive = float(np.linalg.norm(head[0] - hive))
    start_near_hive    = start_dist_to_hive <= START_NEAR_HIVE_MAX

    # If start position is ALREADY inside the v3 hive sphere, extrapolation moot
    auto_passed_extrap = (AUTO_PASS_EXTRAPOLATION_AT_HIVE
                          and start_dist_to_hive <= HIVE_SPHERE_V3)
    hits_sphere_eff = hits_sphere or auto_passed_extrap

    # --- cosine: velocity should point AWAY from hive ---
    outward = head[0] - hive
    outward_norm = np.linalg.norm(outward)
    if outward_norm < 1e-9 or speed_norm < 1e-9:
        cos_outward = 0.0
    else:
        cos_outward = float(np.dot(velocity_unit, outward) / outward_norm)
    # Skip cosine check entirely if we're below the hover-speed threshold
    cos_check_passed = (speed_norm < HOVER_SPEED_MIN) or (cos_outward >= APPROACH_COS_MIN_EXIT)

    hive_exit_v3 = bool(
        is_taking_off
        and hits_sphere_eff
        and start_near_hive
        and cos_check_passed
    )

    return {
        "initial_speed_mean" : initial_speed_mean,
        "initial_speed_min"  : initial_speed_min,
        "start_dist_to_hive" : start_dist_to_hive,
        "backward_t"         : t_close,
        "backward_min_dist"  : dist_close,
        "outward_cos"        : cos_outward,
        "dir_speed_norm"     : speed_norm,
        "auto_passed_extrap" : auto_passed_extrap,
        # per-check booleans for the diagnostic cell
        "pass_speed_exit"    : bool(is_taking_off),
        "pass_extrap_exit"   : bool(hits_sphere_eff),
        "pass_pos_exit"      : bool(start_near_hive),
        "pass_cos_exit"      : bool(cos_check_passed),
        "hive_exit_v3"       : hive_exit_v3,
    }


exit_rows = []
for uid, trk in cleaned_tracks.groupby("uid", sort=False):
    sys_id = uid_to_system.get(uid)
    if sys_id is None or pd.isna(sys_id):
        continue
    hive = HIVE_BY_SYSTEM.get(int(sys_id))
    if hive is None:
        continue

    # v1 (legacy) — first 10 frames within TOL of hive
    coords = _valid_coords(trk)
    if len(coords) < HEAD_FRAMES:
        continue
    head_xyz = coords[:HEAD_FRAMES]
    min_dist_start = float(np.min(np.linalg.norm(head_xyz - hive, axis=1)))
    hive_exit_v1   = min_dist_start <= TOL

    feats = classify_exit_v3(trk, hive)
    if feats is None:
        continue

    exit_rows.append({
        "uid"            : uid,
        "system_id"      : int(sys_id),
        "n_frames"       : int(len(trk)),
        "hive_x"         : hive[0],
        "hive_y"         : hive[1],
        "hive_z"         : hive[2],
        "min_dist_start" : min_dist_start,
        "hive_exit_v1"   : hive_exit_v1,
        **feats,
    })

exit_results = pd.DataFrame(exit_rows)

if not exit_results.empty:
    print(f"Exits v1: {int(exit_results['hive_exit_v1'].sum()):4d}")
    print(f"Exits v3: {int(exit_results['hive_exit_v3'].sum()):4d}")
else:
    print("(no exit candidates)")


In [ ]:

# v3 return detection (symmetric to exit)
# ---------------------------------------------------------------------------


def classify_return_v3(trk, hive):
    coords = _valid_coords(trk)
    if len(coords) < TAIL_FRAMES:
        return None

    tail = coords[-TAIL_FRAMES:]
    dt = 1.0 / FPS

    # --- speed check (bee came to rest at some point in the tail) ---
    tail_velocities = np.diff(tail, axis=0) / dt
    tail_speeds     = np.linalg.norm(tail_velocities, axis=1)
    terminal_speed_mean = float(np.mean(tail_speeds))
    terminal_speed_min  = float(np.min(tail_speeds))
    speed_for_threshold = (terminal_speed_min if USE_MIN_SPEED_FOR_THRESHOLD
                           else terminal_speed_mean)
    is_landing = speed_for_threshold < SPEED_THRESHOLD_LANDING

    # --- direction estimate: use frames BEFORE the tail if track is long enough
    #     (bee was still cruising, vector is cleaner) -----------------------
    n = len(coords)
    dir_start = max(n - TAIL_FRAMES - APPROACH_DIR_FRAME_OFFSET, 0)
    dir_end   = n - TAIL_FRAMES
    if APPROACH_DIR_FRAME_OFFSET > 0 and (dir_end - dir_start) >= 2:
        dir_window = coords[dir_start:dir_end]
        dir_velocities = np.diff(dir_window, axis=0) / dt
    else:
        dir_velocities = tail_velocities
    mean_velocity = dir_velocities.mean(axis=0)
    speed_norm    = float(np.linalg.norm(mean_velocity))
    velocity_unit = mean_velocity / speed_norm if speed_norm >= 1e-9 else np.zeros(3)

    # --- forward extrapolation from tail[-1] ---
    hits_sphere, t_close, dist_close = line_passes_within_radius(
        point=tail[-1], direction=velocity_unit, target=hive,
        radius=HIVE_SPHERE_V3, require_forward=True,
    )

    end_dist_to_hive = float(np.linalg.norm(tail[-1] - hive))
    end_near_hive    = end_dist_to_hive <= END_NEAR_HIVE_MAX

    # If end position is ALREADY inside v3 hive sphere, extrapolation moot
    auto_passed_extrap = (AUTO_PASS_EXTRAPOLATION_AT_HIVE
                          and end_dist_to_hive <= HIVE_SPHERE_V3)
    hits_sphere_eff = hits_sphere or auto_passed_extrap

    # --- cosine: velocity should point TOWARD hive ---
    toward = hive - tail[-1]
    toward_norm = np.linalg.norm(toward)
    if toward_norm < 1e-9 or speed_norm < 1e-9:
        cos_toward = 0.0
    else:
        cos_toward = float(np.dot(velocity_unit, toward) / toward_norm)
    cos_check_passed = (speed_norm < HOVER_SPEED_MIN) or (cos_toward >= APPROACH_COS_MIN_RETURN)

    hive_return_v3 = bool(
        is_landing
        and hits_sphere_eff
        and end_near_hive
        and cos_check_passed
    )

    return {
        "terminal_speed_mean" : terminal_speed_mean,
        "terminal_speed_min"  : terminal_speed_min,
        "end_dist_to_hive"    : end_dist_to_hive,
        "forward_t"           : t_close,
        "forward_min_dist"    : dist_close,
        "toward_cos"          : cos_toward,
        "dir_speed_norm"      : speed_norm,
        "auto_passed_extrap"  : auto_passed_extrap,
        # per-check booleans for the diagnostic cell
        "pass_speed_return"   : bool(is_landing),
        "pass_extrap_return"  : bool(hits_sphere_eff),
        "pass_pos_return"     : bool(end_near_hive),
        "pass_cos_return"     : bool(cos_check_passed),
        "hive_return_v3"      : hive_return_v3,
    }


return_rows = []
for uid, trk in cleaned_tracks.groupby("uid", sort=False):
    sys_id = uid_to_system.get(uid)
    if sys_id is None or pd.isna(sys_id):
        continue
    hive = HIVE_BY_SYSTEM.get(int(sys_id))
    if hive is None:
        continue

    coords = _valid_coords(trk)
    if len(coords) < TAIL_FRAMES:
        continue
    tail_xyz = coords[-TAIL_FRAMES:]
    min_dist_hive = float(np.min(np.linalg.norm(tail_xyz - hive, axis=1)))
    hive_return_v1 = min_dist_hive <= TOL

    feats = classify_return_v3(trk, hive)
    if feats is None:
        continue

    return_rows.append({
        "uid"           : uid,
        "system_id"     : int(sys_id),
        "n_frames"      : int(len(trk)),
        "hive_x"        : hive[0],
        "hive_y"        : hive[1],
        "hive_z"        : hive[2],
        "min_dist_hive" : min_dist_hive,
        "hive_return_v1": hive_return_v1,
        **feats,
    })

return_results = pd.DataFrame(return_rows)

if not return_results.empty:
    print(f"Returns v1: {int(return_results['hive_return_v1'].sum()):4d}")
    print(f"Returns v3: {int(return_results['hive_return_v3'].sum()):4d}")
else:
    print("(no return candidates)")


In [ ]:

# Pair v3 exits with the next v3 return (same system, ≤ MAX_TRIP_DURATION_S)
# ---------------------------------------------------------------------------

exits_v3   = exit_results[exit_results["hive_exit_v3"]].copy()
returns_v3 = return_results[return_results["hive_return_v3"]].copy()

exits_v3["t_start"] = exits_v3["uid"].map(uid_to_start)
returns_v3["t_end"] = returns_v3["uid"].map(uid_to_end)

# Drop any candidates without a usable timestamp
exits_v3   = exits_v3.dropna(subset=["t_start"]).copy()
returns_v3 = returns_v3.dropna(subset=["t_end"]).copy()

exits_v3   = exits_v3.sort_values(["system_id", "t_start"]).reset_index(drop=True)
returns_v3 = returns_v3.sort_values(["system_id", "t_end"]).reset_index(drop=True)

paired = []
matched_return_uids = set()

for _, e in exits_v3.iterrows():
    sys_id  = int(e["system_id"])
    t_start = float(e["t_start"])

    mask = (
        (returns_v3["system_id"] == sys_id)
        & (returns_v3["t_end"] >= t_start)
        & (~returns_v3["uid"].isin(matched_return_uids))
    )
    candidates = returns_v3[mask]
    if candidates.empty:
        continue

    r = candidates.iloc[0]
    duration = float(r["t_end"]) - t_start
    if not np.isfinite(duration) or duration > MAX_TRIP_DURATION_S:
        continue

    paired.append({
        "system_id"       : sys_id,
        "exit_uid"        : e["uid"],
        "return_uid"      : r["uid"],
        "t_start"         : t_start,
        "t_end"           : float(r["t_end"]),
        "trip_duration_s" : duration,
    })
    matched_return_uids.add(r["uid"])

paired_trips = pd.DataFrame(paired)

matched_exit_uids = set(paired_trips["exit_uid"]) if not paired_trips.empty else set()
unmatched_exits   = exits_v3[~exits_v3["uid"].isin(matched_exit_uids)].copy()
unmatched_returns = returns_v3[~returns_v3["uid"].isin(matched_return_uids)].copy()

print(f"Paired trips:      {len(paired_trips):4d}")
print(f"Unmatched exits:   {len(unmatched_exits):4d}")
print(f"Unmatched returns: {len(unmatched_returns):4d}")

if not paired_trips.empty:
    print()
    print("Trip duration (s):")
    print(paired_trips["trip_duration_s"].describe().round(2).to_string())


In [ ]:

# Diagnostics — counts + rejection breakdown for v1-True / v3-False tracks
# ---------------------------------------------------------------------------

def _safe_sum(df, col):
    return int(df[col].sum()) if (not df.empty and col in df.columns) else 0


def _median_dur():
    if paired_trips.empty:
        return None
    return float(paired_trips["trip_duration_s"].median())


summary = pd.DataFrame([
    ("Tracks loaded",                              int(all_tracks["uid"].nunique())),
    ("Tracks dropped (< MIN_TRACK_FRAMES post-clean)", len(dropped_uids)),
    ("Tracks with at least one velocity spike",    int(sum(1 for c in spike_counts if c > 0))),
    ("Spike frames healed (total)",                int(sum(spike_counts))),
    ("Exits v1",                                   _safe_sum(exit_results,   "hive_exit_v1")),
    ("Exits v3",                                   _safe_sum(exit_results,   "hive_exit_v3")),
    ("Returns v1",                                 _safe_sum(return_results, "hive_return_v1")),
    ("Returns v3",                                 _safe_sum(return_results, "hive_return_v3")),
    ("Paired trips",                               len(paired_trips)),
    ("Unmatched exits",                            len(unmatched_exits)),
    ("Unmatched returns",                          len(unmatched_returns)),
    ("Median trip duration (s)",                   _median_dur()),
], columns=["Metric", "Value"])

print(summary.to_string(index=False))


def _rejection_breakdown(df, v1_col, suffix, label):
    """For every track where v1=True but v3=False, count which sub-check failed.
    A track can fail multiple checks (failures aren\'t mutually exclusive)."""
    if df.empty or v1_col not in df.columns:
        return
    rejected = df[df[v1_col] & ~df[f"hive_{label}_v3"]].copy()
    n = len(rejected)
    print(f"\n--- {label.upper()} rejection breakdown (v1=True / v3=False): n={n} ---")
    if n == 0:
        return
    for check, col in [
        ("speed     (was not slow enough at takeoff/landing)", f"pass_speed_{suffix}"),
        ("position  (start/end not close enough to hive)    ", f"pass_pos_{suffix}"),
        ("extrap    (velocity line missed hive sphere)      ", f"pass_extrap_{suffix}"),
        ("cosine    (velocity not pointing toward/away hive)", f"pass_cos_{suffix}"),
    ]:
        if col not in rejected.columns:
            continue
        failed = (~rejected[col]).sum()
        pct = 100 * failed / n if n else 0
        print(f"  failed {check}  {int(failed):4d}  ({pct:5.1f}%)")


_rejection_breakdown(exit_results,   "hive_exit_v1",   "exit",   "exit")
_rejection_breakdown(return_results, "hive_return_v1", "return", "return")


In [ ]:

# Export — everything goes under DATA_DIR/v3_pipeline/ next to flight_tracks.csv
# ---------------------------------------------------------------------------

OUT_DIR = DATA_DIR / "v3_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

exit_results.to_csv(OUT_DIR / "exit_results.csv", index=False)
return_results.to_csv(OUT_DIR / "return_results.csv", index=False)
paired_trips.to_csv(OUT_DIR / "paired_trips.csv", index=False)
unmatched_exits.to_csv(OUT_DIR / "unmatched_exits.csv", index=False)
unmatched_returns.to_csv(OUT_DIR / "unmatched_returns.csv", index=False)
cleaned_tracks.to_csv(OUT_DIR / "flight_tracks_cleaned.csv", index=False)

print(f"Wrote v3 outputs to: {OUT_DIR.resolve()}")
for p in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {p.name:<32s}  {p.stat().st_size/1024:6.1f} KB")
